First import modules and initialize Earth Engine.

In [1]:

# standard modules
import io
import json
import os
from pathlib import Path
import time

# specialized modules
import ee
import geemap
import geopandas as gpd
from pathlib import Path
from tqdm import tqdm

# initialize the Earth Engine module.
ee.Authenticate(auth_mode="localhost")
ee.Initialize(project='nr218-michaelhuggins')

/home/michael/miniforge3/envs/ee/lib/python3.13/site-packages/geemap/conversion.py:23: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  import pkg_resources


Open the AOI vector file and check it on a map.

In [3]:
# read AOIs
# This works whether the notebook is launched from the repo root or from assets/.
this_dir = Path.cwd()
repo_dir = this_dir if (this_dir / 'assets').exists() else this_dir.parent

def first_existing_path(candidates):
    for path in candidates:
        if path.exists():
            return path
    raise FileNotFoundError(f'None of these paths exist: {candidates}')

# Broad analysis/export AOI.
aoi_path = first_existing_path([
    this_dir / 'rio_san_juan_aoi.geojson',
    repo_dir / 'assets' / 'rio_san_juan_aoi.geojson',
])

# Smaller comparison/reference AOI inside the Rio San Juan box.
rio_aoi_path = first_existing_path([
    this_dir / 'rio_indio_aoi.geojson',
    repo_dir / 'untracked_qgis' / 'susques' / 'rio_indio_aoi.geojson',
])

aoi_path, rio_aoi_path

(PosixPath('/home/michael/Work/nr218/assets/rio_san_juan_aoi.geojson'),
 PosixPath('/home/michael/Work/nr218/assets/rio_indio_aoi.geojson'))

In [4]:
aoi = gpd.read_file(aoi_path).to_crs(4326)
rio_aoi = gpd.read_file(rio_aoi_path).to_crs(4326)

gee_json = json.loads(aoi[['geometry']].to_json())
gee_aoi = geemap.geojson_to_ee(gee_json)

gee_rio_json = json.loads(rio_aoi[['geometry']].to_json())
gee_rio_aoi = geemap.geojson_to_ee(gee_rio_json)

# inspect the GeoJSON as an EEObject through geemap.
test_map = geemap.Map(basemap='SATELLITE')
test_map.centerObject(gee_aoi, 9)

# Style the AOIs explicitly so both are visible on the map.
aoi_style = {'color': 'FF0000', 'fillColor': '00000000', 'width': 3}
rio_aoi_style = {'color': 'FFFF00', 'fillColor': '00000000', 'width': 3}
test_map.addLayer(gee_aoi.style(**aoi_style), {}, 'Broad Rio San Juan AOI')
test_map.addLayer(gee_rio_aoi.style(**rio_aoi_style), {}, 'Rio Indio AOI')

test_map


Skipping field bbox: unsupported OGR type: 3


Map(center=[10.990011493745822, -83.83500000000112], controls=(WidgetControl(options=['position', 'transparent…

Get the vertices of the broad AOI and Rio Indio AOI to use in Earth Engine filters.


In [5]:
# get extents for both AOIs
def bounds_to_verts(gdf):
    minx, miny, maxx, maxy = gdf.total_bounds
    return [
        [float(minx), float(miny)],
        [float(minx), float(maxy)],
        [float(maxx), float(maxy)],
        [float(maxx), float(miny)],
        [float(minx), float(miny)],
    ]

# Broad box from the original Rio San Juan AOI.
verts = bounds_to_verts(aoi)

# Smaller AOI used for Rio Indio wet-season composites.
rio_verts = bounds_to_verts(rio_aoi)

verts, rio_verts


([[-84.05, 10.88],
  [-84.05, 11.1],
  [-83.62, 11.1],
  [-83.62, 10.88],
  [-84.05, 10.88]],
 [[-83.79898991031392, 10.900770179372204],
  [-83.79898991031392, 11.025720852017939],
  [-83.62053026905832, 11.025720852017939],
  [-83.62053026905832, 10.900770179372204],
  [-83.79898991031392, 10.900770179372204]])

# Wet-season composite periods

Use these periods to download annual wet-season composites for the Rio Indio AOI. 


In [10]:
# wet season for the Caribbean side of Nicaragua.
# end dates are exclusive, so November composites end on December 1.
WET_START_MONTH = 5
WET_END_MONTH = 11

composite_periods = {
    'landsat_5_7': {
        'start_year': 2000,
        'end_year': 2011,
        'start_month': WET_START_MONTH,
        'end_month': WET_END_MONTH,
        'scale': 30,
        'short_name': 'l57',
    },
    'landsat_8_9': {
        'start_year': 2013,
        'end_year': 2025,
        'start_month': WET_START_MONTH,
        'end_month': WET_END_MONTH,
        'scale': 30,
        'short_name': 'l89',
    },
    'sentinel_2': {
        'start_year': 2018,
        'end_year': 2025,
        'start_month': WET_START_MONTH,
        'end_month': WET_END_MONTH,
        'scale': 10,
        'short_name': 's2',
    },
}

# Use this region for Rio Indio downloads.
rio_export_region = gee_rio_aoi.geometry()


def wet_season_date_range(year):
    start = ee.Date.fromYMD(year, WET_START_MONTH, 1)
    end = ee.Date.fromYMD(year, WET_END_MONTH, 1).advance(1, 'month')
    return start, end


# Quick check: these are the image years that will be requested for each sensor group.
for sensor, period in composite_periods.items():
    years = list(range(period['start_year'], period['end_year'] + 1))
    print(sensor, years[0], 'to', years[-1], f"({len(years)} composites)")


landsat_5_7 2000 to 2011 (12 composites)
landsat_8_9 2013 to 2025 (13 composites)
sentinel_2 2018 to 2025 (8 composites)


## Export seasonal annual composites to Google Drive.

The helper below works for either season. Use `queue_drive_exports(season='wet', sensor='sentinel_2')` or `queue_drive_exports(season='dry', sensor='sentinel_2')`. Composites use a weighted quality mosaic instead of a median composite.


In [ ]:
COMMON_BANDS = ['blue', 'green', 'red', 'nir', 'swir1', 'swir2']
DRIVE_FOLDERS = {
    'wet': 'nicaragua_wet_season',
    'dry': 'nicaragua_dry_season',
}


def mask_landsat_sr(image):
    qa = image.select('QA_PIXEL')
    cloud_shadow = 1 << 4
    clouds = 1 << 3
    mask = qa.bitwiseAnd(cloud_shadow).eq(0).And(qa.bitwiseAnd(clouds).eq(0))
    return image.updateMask(mask)


def prep_landsat_5_7(image):
    optical = image.select(
        ['SR_B1', 'SR_B2', 'SR_B3', 'SR_B4', 'SR_B5', 'SR_B7'],
        COMMON_BANDS,
    ).multiply(0.0000275).add(-0.2)
    return optical.copyProperties(image, image.propertyNames())


def prep_landsat_8_9(image):
    optical = image.select(
        ['SR_B2', 'SR_B3', 'SR_B4', 'SR_B5', 'SR_B6', 'SR_B7'],
        COMMON_BANDS,
    ).multiply(0.0000275).add(-0.2)
    return optical.copyProperties(image, image.propertyNames())


def mask_s2_clouds(image):
    qa = image.select('QA60')
    clouds = 1 << 10
    cirrus = 1 << 11
    mask = qa.bitwiseAnd(clouds).eq(0).And(qa.bitwiseAnd(cirrus).eq(0))
    optical = image.select(['B2', 'B3', 'B4', 'B8', 'B11', 'B12'], COMMON_BANDS)
    return optical.updateMask(mask).divide(10000).copyProperties(image, image.propertyNames())


def get_season_config(season):
    if season == 'wet':
        return {
            'periods': composite_periods,
            'date_range': wet_season_date_range,
            'folder': DRIVE_FOLDERS['wet'],
            'region': rio_export_region,
            'suffix': 'wet_season',
        }
    if season == 'dry':
        return {
            'periods': dry_composite_periods,
            'date_range': dry_season_date_range,
            'folder': DRIVE_FOLDERS['dry'],
            'region': rio_dry_export_region,
            'suffix': 'dry_season',
        }
    raise ValueError(f'Unknown season: {season}')


def build_sensor_collection(sensor, start, end, region):
    if sensor == 'landsat_5_7':
        return (
            ee.ImageCollection('LANDSAT/LT05/C02/T1_L2')
            .merge(ee.ImageCollection('LANDSAT/LE07/C02/T1_L2'))
            .filterBounds(region)
            .filterDate(start, end)
            .map(mask_landsat_sr)
            .map(prep_landsat_5_7)
        )
    if sensor == 'landsat_8_9':
        return (
            ee.ImageCollection('LANDSAT/LC08/C02/T1_L2')
            .merge(ee.ImageCollection('LANDSAT/LC09/C02/T1_L2'))
            .filterBounds(region)
            .filterDate(start, end)
            .map(mask_landsat_sr)
            .map(prep_landsat_8_9)
        )
    if sensor == 'sentinel_2':
        return (
            ee.ImageCollection('COPERNICUS/S2_SR_HARMONIZED')
            .filterBounds(region)
            .filterDate(start, end)
            .filter(ee.Filter.lt('CLOUDY_PIXEL_PERCENTAGE', 35))
            .map(mask_s2_clouds)
        )
    raise ValueError(f'Unknown sensor: {sensor}')


def add_mosaic_score(image, start, end, cloud_property):
    season_mid = start.advance(end.difference(start, 'day').divide(2), 'day')
    half_window = end.difference(start, 'day').divide(2)

    days_from_mid = image.date().difference(season_mid, 'day').abs()
    date_score = ee.Image.constant(1).subtract(ee.Image.constant(days_from_mid).divide(half_window)).clamp(0, 1)

    cloud_value = ee.Number(
        ee.Algorithms.If(
            image.propertyNames().contains(cloud_property),
            image.get(cloud_property),
            0,
        )
    )
    cloud_score = ee.Image.constant(1).subtract(cloud_value.divide(100)).clamp(0, 1)

    score = date_score.multiply(0.4).add(cloud_score.multiply(0.6)).rename('score')
    score = score.updateMask(image.select(COMMON_BANDS[0]).mask())
    return image.addBands(score)


def build_season_composite(sensor, year, season='wet', region=None):
    config = get_season_config(season)
    if region is None:
        region = config['region']
    start, end = config['date_range'](year)
    cloud_property = 'CLOUDY_PIXEL_PERCENTAGE' if sensor == 'sentinel_2' else 'CLOUD_COVER'

    collection = build_sensor_collection(sensor, start, end, region)
    scored = collection.map(lambda image: add_mosaic_score(image, start, end, cloud_property))

    return scored.qualityMosaic('score').select(COMMON_BANDS).clip(region)


def queue_drive_exports(season='wet', sensor='sentinel_2', folder=None, region=None):
    config = get_season_config(season)
    periods = config['periods']
    period = periods[sensor]
    folder = folder or config['folder']
    if region is None:
        region = config['region']
    tasks = []

    for year in range(period['start_year'], period['end_year'] + 1):
        image = build_season_composite(sensor, year, season=season, region=region)
        prefix = f"{period['short_name']}_{year}_{config['suffix']}"
        task = ee.batch.Export.image.toDrive(
            image=image,
            description=prefix,
            folder=folder,
            fileNamePrefix=prefix,
            region=region,
            scale=period['scale'],
            fileFormat='GeoTIFF',
            maxPixels=1e13,
        )
        task.start()
        tasks.append(task)
        print(f'Started export: {prefix}')

    return tasks


# Examples:
# wet_tasks = queue_drive_exports(season='wet', sensor='sentinel_2')
# dry_tasks = queue_drive_exports(season='dry', sensor='sentinel_2')



In [ ]:
tasks = []

for sensor in composite_periods.keys():
    tasks.extend(queue_drive_exports(season='wet', sensor=sensor))

print(f'Queued {len(tasks)} wet-season export tasks.')


In [ ]:
flat_tasks = []
for task in tasks:
    if isinstance(task, list):
        flat_tasks.extend(task)
    else:
        flat_tasks.append(task)

seen_done = set()
terminal_states = {'COMPLETED', 'FAILED', 'CANCELLED'}
not_done = True

while not_done:
    for task in flat_tasks:
        status = task.status()
        description = status.get('description', task.id)
        state = status.get('state', 'UNKNOWN')

        if state in terminal_states and description not in seen_done:
            print(description, state)
            seen_done.add(description)

    not_done = any(task.status().get('state') not in terminal_states for task in flat_tasks)

    if not_done:
        print('Waiting 3 minutes before checking again...')
        time.sleep(180)

print('All tasks are done.')

# Dry-season composite periods

Use these periods to download annual dry-season composites for the Rio Indio AOI. 


In [7]:
# dry season for the Caribbean side of Nicaragua.
# end dates are exclusive, so April composites end on May 1.
DRY_START_MONTH = 2
DRY_END_MONTH = 4

dry_composite_periods = {
    'landsat_5_7': {
        'start_year': 2000,
        'end_year': 2011,
        'start_month': DRY_START_MONTH,
        'end_month': DRY_END_MONTH,
        'scale': 30,
        'short_name': 'l57',
    },
    'landsat_8_9': {
        'start_year': 2013,
        'end_year': 2025,
        'start_month': DRY_START_MONTH,
        'end_month': DRY_END_MONTH,
        'scale': 30,
        'short_name': 'l89',
    },
    'sentinel_2': {
        'start_year': 2018,
        'end_year': 2025,
        'start_month': DRY_START_MONTH,
        'end_month': DRY_END_MONTH,
        'scale': 10,
        'short_name': 's2',
    },
}

# Use this region for Rio Indio downloads.
rio_dry_export_region = gee_rio_aoi.geometry()


def dry_season_date_range(year):
    start = ee.Date.fromYMD(year, DRY_START_MONTH, 1)
    end = ee.Date.fromYMD(year, DRY_END_MONTH, 1).advance(1, 'month')
    return start, end


# Quick check: these are the image years that will be requested for each sensor group.
for sensor, period in dry_composite_periods.items():
    years = list(range(period['start_year'], period['end_year'] + 1))
    print(sensor, years[0], 'to', years[-1], f"({len(years)} composites)")

landsat_5_7 2000 to 2011 (12 composites)
landsat_8_9 2013 to 2025 (13 composites)
sentinel_2 2018 to 2025 (8 composites)


## Export dry-season annual composites to Google Drive.

Use the same helper cell above, but pass `season='dry'`. For example: `queue_drive_exports(season='dry', sensor='sentinel_2')`.


In [ ]:
# Dry-season exports use the shared queue_drive_exports() helper defined above.
# Example:
# dry_tasks = queue_drive_exports(season='dry', sensor='sentinel_2')


In [ ]:
tasks = []

for sensor in dry_composite_periods.keys():
    tasks.extend(queue_drive_exports(season='dry', sensor=sensor))

print(f'Queued {len(tasks)} dry-season export tasks.')


In [ ]:
flat_tasks = []
for task in tasks:
    if isinstance(task, list):
        flat_tasks.extend(task)
    else:
        flat_tasks.append(task)

seen_done = set()
terminal_states = {'COMPLETED', 'FAILED', 'CANCELLED'}
not_done = True

while not_done:
    for task in flat_tasks:
        status = task.status()
        description = status.get('description', task.id)
        state = status.get('state', 'UNKNOWN')

        if state in terminal_states and description not in seen_done:
            print(description, state)
            seen_done.add(description)

    not_done = any(task.status().get('state') not in terminal_states for task in flat_tasks)

    if not_done:
        print('Waiting 3 minutes before checking again...')
        time.sleep(180)

print('All tasks are done.')

Waiting 3 minutes before checking again...
Waiting 3 minutes before checking again...
l57_2000_dry_season COMPLETED
l57_2001_dry_season COMPLETED
l57_2002_dry_season COMPLETED
l57_2003_dry_season COMPLETED
l57_2004_dry_season COMPLETED
l57_2005_dry_season COMPLETED
l57_2006_dry_season COMPLETED
l57_2007_dry_season COMPLETED
l57_2008_dry_season COMPLETED
l57_2009_dry_season COMPLETED
l57_2010_dry_season COMPLETED
l57_2011_dry_season COMPLETED
l89_2013_dry_season COMPLETED
Waiting 3 minutes before checking again...


for task in ee.batch.Task.list():
    status = task.status()
    desc = status.get("description", "")
    state = status.get("state", "")

    if state in ["READY", "RUNNING"] and ("wet_season" in desc or "dry_season" in desc):
        print("Cancelling", desc, state)
        task.cancel()